# LangRank + URIEL+ (Config B: Standard PCA) - Reproducible Pipeline

Runs the README workflow end to end, then checks the results against the reference run:

1. Set up the environment (LightGBM check)
2. Place the URIEL+ data (unzipped from `standard_pca.zip` and `data.zip` automatically)
3. Calculate distances (`distances/calculate_distances.py`)
4. Update the experiment CSVs (`distances/replace_distances.py`)
5. Run the 8 experiments (DEP/EL/MT/POS × `all`/`lang`)
6. Present results (table of average NDCG@3)
7. Package everything generated into `standard_pca_reproduction.zip`

**Use:** open this notebook inside the `URIELPlus-LangRank` folder (or set the `LANGRANK_DIR` environment variable) and Run All. Windows only (the repo scripts use backslash paths).

**Time:** a fresh run takes about 2.5 hours, almost all of it Step 3 (distances for 5,754 language pairs, ~1.5 s each). Keep the machine awake and the kernel running; if it is interrupted, rerun and Step 3 restarts from scratch. Step 3 is only skipped when complete distance CSVs already exist. Everything else takes a few minutes.

**Output:** `results/scores.txt` (per-fold NDCG@3 + averages) and `results/timing_logs/` (timing logs written by the repo scripts). The last cell can delete everything this notebook generates.

## 0. Setup

In [1]:
import os, re, sys, time, shutil, subprocess, zipfile, importlib.util
from pathlib import Path

TASKS = ["dep", "el", "mt", "pos"]
MODES = ["all", "lang"]      # LangRank (all features) / LangRank (lang feats)
FORCE = False                # True -> recompute distances even if complete CSVs exist

assert sys.version_info >= (3, 10), "Python 3.10+ is required"

def find_langrank_dir():
    """LANGRANK_DIR env var, else walk up from the notebook's folder."""
    env = os.environ.get("LANGRANK_DIR")
    if env:
        return Path(env).expanduser().resolve()
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base, base / "URIELPlus-LangRank"):
            if (cand / "distances" / "calculate_distances.py").exists():
                return cand
    raise FileNotFoundError("URIELPlus-LangRank repo not found. Open this notebook inside it or set LANGRANK_DIR.")

LANGRANK_DIR = find_langrank_dir()
DIST_DIR    = LANGRANK_DIR / "distances"
DB_DIR      = DIST_DIR / "urielplus" / "database"          # where URIEL+ reads its .npz files
INPUT_DIR   = LANGRANK_DIR / "experiment_csvs" / "URIEL"   # original experiment CSVs (language pairs)
CSV_DIR     = LANGRANK_DIR / "src" / "csv_datasets"        # updated experiment CSVs (output of Step 4)
RESULTS_DIR = LANGRANK_DIR / "results"
TIMING_DIR  = RESULTS_DIR / "timing_logs"
TIMING_DIR.mkdir(parents=True, exist_ok=True)
NPZ = ["family_features.npz", "features.npz", "script_features.npz", "geocoord_features.npz"]
DIST_COLS = ["GENETIC", "SYNTACTIC", "FEATURAL", "PHONOLOGICAL", "INVENTORY", "GEOGRAPHIC", "MORPHOLOGICAL", "SCRIPT"]
PYTHON = sys.executable

needed = ([DIST_DIR / "calculate_distances.py", DIST_DIR / "replace_distances.py",
           DB_DIR / "urielplus_csvs" / "uriel_glottocode_map.csv"]
          + [LANGRANK_DIR / "src" / "langrank_models" / f"{t}.py" for t in TASKS]
          + [INPUT_DIR / f"{t}.csv" for t in TASKS])
missing = [str(p.relative_to(LANGRANK_DIR)) for p in needed if not p.exists()]
assert not missing, f"Missing repo files: {missing}"

def run(cmd, hide=None, tick=None):
    """Run a command from the repo root, stream its output, return it. Raises on failure.
    hide: regex of lines not to display. tick: regex of lines to count, shown as a progress line every 250."""
    print("$", " ".join(map(str, cmd)))
    hide, tick = (re.compile(hide) if hide else None), (re.compile(tick) if tick else None)
    env = {**os.environ, "LOKY_MAX_CPU_COUNT": str(os.cpu_count() or 1)}   # silences a harmless joblib warning on Windows
    t0, n, lines = time.time(), 0, []
    proc = subprocess.Popen(cmd, cwd=LANGRANK_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        lines.append(line)
        if tick and tick.search(line):
            n += 1
            if n % 250 == 0:
                print(f"  ... {n} done ({(time.time() - t0) / 60:.1f} min)")
        if not (hide and hide.search(line)):
            print(line, end="")
    rc = proc.wait()
    print(f"-> exit {rc} in {time.time() - t0:.1f}s")
    if rc != 0:
        raise RuntimeError(f"Command failed ({rc}): {cmd}")
    return "".join(lines)

def collect_timing_logs():
    """The repo scripts write timing logs next to themselves; gather them in results/timing_logs/."""
    for p in [*DIST_DIR.glob("*_timing_log.txt"), *LANGRANK_DIR.glob("langrank_*_timing_log_*.txt")]:
        os.replace(p, TIMING_DIR / p.name)

print("Repo:", LANGRANK_DIR)

Repo: c:\Users\User\Downloads\University\Fourth Year\Fall Semester\CSCI 3052U\Group Project\csci3052u-ml-project\URIELPlus-LangRank


## 1. Environment
Installs the LangRank requirements if a key package is missing, then checks LightGBM with a tiny ranking fit. On an older Anaconda (numpy 1.24) the latest LightGBM (4.7) crashes with `access violation`; in that case the cell installs an older compatible LightGBM + scikit-learn pair (per-user install) and re-tests.

In [2]:
if [m for m in ("pandas", "numpy", "lightgbm", "sklearn") if importlib.util.find_spec(m) is None]:
    run([PYTHON, "-m", "pip", "install", "-r", str(LANGRANK_DIR / "requirements" / "langrank-requirements.txt")])

SMOKE = """
import numpy as np, pandas as pd, sklearn, lightgbm as lgb
from lightgbm import LGBMRanker
rng = np.random.default_rng(0)
X = pd.DataFrame(rng.random((60, 4)), columns=list("abcd")); y = pd.Series(rng.integers(0, 5, 60)).astype(float)
r = LGBMRanker(objective="lambdarank", n_estimators=5, num_leaves=4, min_data_in_leaf=2, verbose=-1)
r.fit(X, y, group=[20, 20, 20], eval_set=[(X[:20], y[:20])], eval_group=[[20]], eval_at=[3])
print("SMOKE_OK lightgbm", lgb.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)
"""

def lightgbm_works():
    r = subprocess.run([PYTHON, "-c", SMOKE], capture_output=True, text=True)
    print((r.stdout or r.stderr).strip()[-300:])
    return next((l for l in r.stdout.splitlines() if l.startswith("SMOKE_OK")), None)

ENV_LINE = lightgbm_works()
if not ENV_LINE:
    user = [] if sys.prefix != sys.base_prefix else ["--user"]      # --user is not allowed inside a venv
    for lgbm, sk in [("4.5.0", "1.5.2"), ("4.6.0", "1.6.1"), ("4.3.0", "1.3.2")]:
        print(f"LightGBM crashed; trying lightgbm=={lgbm} + scikit-learn=={sk}")
        if subprocess.call([PYTHON, "-m", "pip", "install", *user, f"lightgbm=={lgbm}", f"scikit-learn=={sk}"]) == 0:
            ENV_LINE = lightgbm_works()
            if ENV_LINE: break
    assert ENV_LINE, "LightGBM still fails after trying older versions."

SMOKE_OK lightgbm 4.6.0 | numpy 1.26.4 | scikit-learn 1.6.1 | pandas 2.2.3


## 2. Standard PCA Data
This cell does the data handling. It reads `data.zip` (in the repo folder or the folder above it) and copies the `family_features`, `script_features`, and `geocoord_features` URIEL+ `.npz` files into `distances/urielplus/database/`. Those files (`family_features`, `script_features`, `geocoord_features`) are the **top-level** ones in `data.zip`, not the ones in `non_imputed_data/`. The `features` URIEL+ `.npz` file is copied from `standard_pca.zip`.

It first checks whether the right data is already in place by computing the Czech-Russian row and comparing it with the reference (`dep_distances.csv`, first row). If it doesn't match, it calls `place_data()` and checks again.

To supply the data some other way, only `place_data()` needs to change; the check after it confirms the result.

In [3]:
import json
PROBE_PAIR = ["czec1258", "russ1263"]
PROBE_REF  = [0.4788, 0.4617, 0.4072, 0.2301, 0.3635, 0.0995, 0.3508, 0.369]
PROBE_CODE = """
import json
from urielplus import urielplus as uriel
u = uriel.URIELPlus()
D = ["genetic","syntactic","featural","phonological","inventory","geographic","morphological","script"]
try:
    print("PROBE_RESULT " + json.dumps(u.new_distance(D, %r)))
except Exception as e:
    print("PROBE_ERROR " + str(e).splitlines()[0])
""" % (PROBE_PAIR,)

def data_ok():
    r = subprocess.run([PYTHON, "-c", PROBE_CODE], cwd=DIST_DIR, capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith("PROBE_RESULT "):
            vals = json.loads(line[len("PROBE_RESULT "):])
            return all(abs(v - ref) <= 6e-4 for v, ref in zip(vals, PROBE_REF))
    return False

def place_data():
    """Copy the four URIEL+ .npz files into DB_DIR: features.npz from standard_pca.zip, the rest from data.zip."""
    find = lambda n: next((d / n for d in (LANGRANK_DIR, LANGRANK_DIR.parent) if (d / n).exists()), None)
    zpath, ppath = find("data.zip"), find("standard_pca.zip")
    assert zpath, "data.zip not found next to or above the repo folder"
    assert ppath, "standard_pca.zip not found next to or above the repo folder"
    DB_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z, zipfile.ZipFile(ppath) as zp:
        for name in NPZ:
            if name == "features.npz":
                zf, member = zp, next(m for m in zp.namelist() if m.split("/")[-1] == "features.npz")
            else:
                zf, member = z, f"data/{name}"
            with zf.open(member) as src, open(DB_DIR / name, "wb") as dst:
                shutil.copyfileobj(src, dst)
    print("copied", NPZ, "from", zpath, "and features.npz from", ppath)

if data_ok():
    print("URIEL+ data already in place.")
else:
    place_data()
    assert data_ok(), "URIEL+ data check failed: the Czech-Russian row does not match the reference."
    print("URIEL+ data placed and verified.")

copied ['family_features.npz', 'features.npz', 'script_features.npz', 'geocoord_features.npz'] from c:\Users\User\Downloads\University\Fourth Year\Fall Semester\CSCI 3052U\Group Project\csci3052u-ml-project\data.zip and features.npz from c:\Users\User\Downloads\University\Fourth Year\Fall Semester\CSCI 3052U\Group Project\csci3052u-ml-project\standard_pca.zip
URIEL+ data placed and verified.


## 3. Distances
Runs `distances/calculate_distances.py` (~1.5 s per language pair; ~2 h in total). Skipped when all four `*_distances.csv` files are complete, i.e. have one row per row of the matching experiment CSV. Incomplete files from an interrupted run are deleted first.

In [4]:
import pandas as pd

def distances_complete(t):
    out = DIST_DIR / f"{t}_distances.csv"
    return out.exists() and len(pd.read_csv(out)) == len(pd.read_csv(INPUT_DIR / f"{t}.csv"))

status = {t: distances_complete(t) for t in TASKS}
print("Distance CSVs complete:", status)
if all(status.values()) and not FORCE:
    print("Skipping calculation (FORCE = True to recompute).")
else:
    for t in TASKS:
        (DIST_DIR / f"{t}_distances.csv").unlink(missing_ok=True)
    run([PYTHON, "distances/calculate_distances.py"], hide=r" - root - INFO - |^\d\.\d{4},", tick=r"^\d\.\d{4},")
    collect_timing_logs()
    assert all(distances_complete(t) for t in TASKS), "Distance calculation did not produce complete CSVs"

Distance CSVs complete: {'dep': False, 'el': False, 'mt': False, 'pos': False}
$ c:\Users\User\anaconda3\envs\urielplus\python.exe distances/calculate_distances.py
[Initialize URIELPlus] step time: 0.34s | total elapsed: 0.34s
[Load mappings] step time: 0.02s | total elapsed: 0.36s
  ... 250 done (5.2 min)
  ... 500 done (10.3 min)
  ... 750 done (15.3 min)
[Calculate distances for dep] step time: 1062.28s | total elapsed: 1062.64s
  ... 1000 done (20.4 min)
  ... 1250 done (25.5 min)
[Calculate distances for el] step time: 588.65s | total elapsed: 1651.29s
  ... 1500 done (30.7 min)
  ... 1750 done (36.0 min)
  ... 2000 done (41.4 min)
  ... 2250 done (46.7 min)
  ... 2500 done (52.1 min)
  ... 2750 done (57.4 min)
  ... 3000 done (62.8 min)
  ... 3250 done (68.1 min)
  ... 3500 done (73.5 min)
  ... 3750 done (78.8 min)
  ... 4000 done (84.1 min)
[Calculate distances for mt] step time: 3661.05s | total elapsed: 5312.34s
  ... 4250 done (89.4 min)
  ... 4500 done (94.8 min)
  ... 4750

## 4. Update Experiment CSVs
Runs `distances/replace_distances.py`, which writes `src/csv_datasets/{dep,el,mt,pos}.csv`, then checks row counts and missing values.

In [5]:
run([PYTHON, "distances/replace_distances.py"])
collect_timing_logs()
for t in TASKS:
    df = pd.read_csv(CSV_DIR / f"{t}.csv")
    assert len(df) == len(pd.read_csv(INPUT_DIR / f"{t}.csv")), f"{t}: row count mismatch"
    assert not df[DIST_COLS].isna().any().any(), f"{t}: missing distance values"
    print(f"{t}: {len(df)} rows OK")

$ c:\Users\User\anaconda3\envs\urielplus\python.exe distances/replace_distances.py
[Update dep experiment csv] step time: 0.04s | total elapsed: 0.04s
[Update el experiment csv] step time: 0.02s | total elapsed: 0.06s
[Update mt experiment csv] step time: 0.10s | total elapsed: 0.16s
[Update pos experiment csv] step time: 0.05s | total elapsed: 0.21s

All done. Total time: 0.21s
-> exit 0 in 1.6s
dep: 870 rows OK
el: 477 rows OK
mt: 2862 rows OK
pos: 1545 rows OK


## 5. Experiments
Runs `src/langrank_models/{task}.py --mode {all|lang}` for each task and collects the per-fold NDCG@3 scores into `results/scores.txt` (same layout as the reference `scores.txt`).

In [6]:
NAMES = {"dep": "Dependency Parsing (DEP)", "el": "Entity Linking (EL)",
         "mt": "Machine Translation (MT)", "pos": "Part of Speech (POS) Tagging"}
LABEL = {"all": "LangRank (all)", "lang": "LangRank (lang feats)"}

averages, blocks = {}, [f"# environment: {ENV_LINE}\n"]
for t in TASKS:
    for m in MODES:
        out = run([PYTHON, f"src/langrank_models/{t}.py", "--mode", m], hide=r"^\[Leave-one-out fold")
        scores = re.findall(r"^\[[0-9eE.,\s\-]+\]\s*$", out, re.M)[-1].strip()
        avg = float(re.search(r"Average NDCG@3:\s*([0-9.]+)", out).group(1))
        averages[(t, m)] = avg
        blocks.append(f"{NAMES[t]} Experiments - {LABEL[m]}\n{scores}\nAverage NDCG@3: {avg}\n")
(RESULTS_DIR / "scores.txt").write_text("\n".join(blocks), encoding="utf-8")
collect_timing_logs()
print("Wrote", RESULTS_DIR / "scores.txt")

$ c:\Users\User\anaconda3\envs\urielplus\python.exe src/langrank_models/dep.py --mode all
[Load data] step time: 0.00s | total elapsed: 0.00s
[Assign relevances] step time: 0.10s | total elapsed: 0.10s
[0.2861, 0.4785, 0.9359, 0.9038, 0.7647, 0.8342, 0.801, 1.0, 1.0, 0.8651, 0.905, 0.3823, 0.6951, 0.9304, 0.4065, 0.9038, 0.2795, 0.5082, 0.5239, 0.8288, 0.9933, 1.0, 0.9746, 0.8718, 0.9238, 0.9292, 0.9812, 0.9171, 0.9492, 0.2541]
Average NDCG@3: 76.8
[Compute average NDCG@3] step time: 0.00s | total elapsed: 3.74s

All done. Total time: 3.74s
-> exit 0 in 6.9s
$ c:\Users\User\anaconda3\envs\urielplus\python.exe src/langrank_models/dep.py --mode lang
[Load data] step time: 0.02s | total elapsed: 0.02s
[Assign relevances] step time: 0.11s | total elapsed: 0.12s
[0.3714, 0.4828, 0.9746, 0.8796, 1.0, 0.8651, 0.8409, 0.9558, 0.7967, 0.7967, 0.8596, 0.0, 0.4065, 0.905, 0.2244, 0.559, 0.5723, 0.5989, 0.8663, 0.5723, 0.9933, 1.0, 1.0, 0.7526, 1.0, 0.9492, 0.778, 0.9038, 1.0, 0.0]
Average NDCG@3:

## 6. Results Table
Parses `results/scores.txt` into a task × mode table of average NDCG@3 (with the mean over the four tasks), shows it, and saves it as `results/results_table.csv`. Needs section 0 to have been run, and `scores.txt` to exist (section 5).

In [ ]:
import pandas as pd
from IPython.display import display

NAMES = {"dep": "Dependency Parsing (DEP)", "el": "Entity Linking (EL)",
         "mt": "Machine Translation (MT)", "pos": "Part of Speech (POS) Tagging"}
LABEL = {"all": "LangRank (all)", "lang": "LangRank (lang feats)"}

text = (RESULTS_DIR / "scores.txt").read_text(encoding="utf-8")
pattern = re.compile(r"^(.+?) Experiments - (.+)\n\[.*\]\nAverage NDCG@3:\s*([0-9.]+)", re.M)
found = {(name, label): float(avg) for name, label, avg in pattern.findall(text)}

rows = {}
for t in TASKS:
    for m in MODES:
        assert (NAMES[t], LABEL[m]) in found, f"{t}/{m} not found in scores.txt"
    rows[t.upper()] = {LABEL[m]: found[(NAMES[t], LABEL[m])] for m in MODES}

results_df = pd.DataFrame(rows).T
results_df.loc["Mean (4 tasks)"] = results_df.loc[[t.upper() for t in TASKS]].mean()
results_df.index.name = "Task"
results_df = results_df.round(1)
results_df.to_csv(RESULTS_DIR / "results_table.csv")
print("Average NDCG@3 (x100)")
display(results_df)

## 7. Package Results
Bundles everything this run generated into `standard_pca_reproduction.zip` (in the repo folder):

| In the zip | Source |
|---|---|
| `distances/{dep,el,mt,pos}_distances.csv` | `distances/` |
| `experiment_csvs/{dep,el,mt,pos}.csv` | `src/csv_datasets/` |
| `timing_logs/*.txt` | `results/timing_logs/` |
| `scores.txt`, `results_table.csv` | `results/` |

The zip is written and checked (file list and sizes) **before** anything is deleted. With `MOVE = True` the originals are then removed, which stops a later config from reusing these distance CSVs (Step 3 skips the calculation when complete CSVs exist). Set `MOVE = False` to keep the originals in place.

In [ ]:
MOVE = True   # True -> delete the originals once the zip is checked; False -> leave them in place
ZIP_PATH = LANGRANK_DIR / "standard_pca_reproduction.zip"

collect_timing_logs()

files = {}   # file on disk -> path inside the zip
for t in TASKS:
    files[DIST_DIR / f"{t}_distances.csv"] = f"distances/{t}_distances.csv"
    files[CSV_DIR / f"{t}.csv"] = f"experiment_csvs/{t}.csv"
files[RESULTS_DIR / "scores.txt"] = "scores.txt"
files[RESULTS_DIR / "results_table.csv"] = "results_table.csv"
for p in sorted(TIMING_DIR.glob("*.txt")):
    files[p] = f"timing_logs/{p.name}"

missing = [str(p) for p in files if not p.exists()]
assert not missing, f"Cannot package, missing files (run sections 3-6 first): {missing}"
assert any(a.startswith("timing_logs/") for a in files.values()), "No timing logs found in results/timing_logs/"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for src, arc in files.items():
        z.write(src, arc)

with zipfile.ZipFile(ZIP_PATH) as z:
    assert z.testzip() is None, "Zip is corrupt"
    assert sorted(z.namelist()) == sorted(files.values()), "Zip contents do not match the file list"
    for src, arc in files.items():
        assert z.getinfo(arc).file_size == src.stat().st_size, f"Size mismatch: {arc}"
    print(f"Wrote {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB), {len(files)} files:")
    for info in z.infolist():
        print(f"  {info.filename}  ({info.file_size / 1e3:.1f} KB)")

if MOVE:
    for src in files:
        src.unlink()
    print(f"Removed {len(files)} original files (now only inside the zip).")
else:
    print("MOVE = False: originals left in place.")